ReACT agent architecture is mostly used when 2 or more queries are made to the llm in a single prompt

In [35]:
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from langchain.agents import create_agent
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START,END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from tavily import TavilyClient
from langchain.tools import tool
from typing import TypedDict, Annotated
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

In [36]:
load_dotenv()

True

In [37]:
llm = init_chat_model(model="google_genai:gemini-3.5-flash")

In [38]:
tavily_client = TavilyClient()

@tool
def tavily_search(query: str) -> dict:
    """Search the web for up-to-date information, news, facts, and answers."""
    return tavily_client.search(query=query)


In [39]:
#custom fucntion
@tool
def multiply(a:int, b:int) -> int:
    """multiply a and b
    
    args:
    a: first int
    b: second int
    
    returns:
    int: output int"""
    return a*b

In [40]:
tools = [tavily_search, multiply]
llm_with_tools = llm.bind_tools(tools)
 

In [41]:
llm_with_tools 

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.5-flash', temperature=1.0, client=<google.genai.client.Client object at 0x0000022A9B2171A0>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'Search the web for up-to-date information, news, facts, and answer

In [42]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

In [ ]:
#node functionality
def tool_calling_llm(state:State):
    #we are giving this system prompt so that the model does not loop back in aimessage and tool call which it sometimes does which burns token limit
    system = SystemMessage(content="You are a helpful assistant. Use tools when needed but avoid redundant searches. One search should be sufficient for most questions.")
    return {"messages":[llm_with_tools.invoke([system] + state['messages'])]}

In [44]:
#graph
builder =StateGraph(State)

In [45]:
#nodes
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools",ToolNode(tools))

adding memory checkpointer

In [46]:
memory = MemorySaver()

In [47]:
config = {"configurable":{"thread_id":"1"}}

Implementing react architecture here

In [48]:
#edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    #if the latest message from assistant is a tool call -> tool_condition results to tools
    #If the latest message from assitant is not a tool call -> tool_conditions routes to END"""
    tools_condition
)
builder.add_edge("tools","tool_calling_llm")

compiling graph with memory

In [49]:
graph = builder.compile(checkpointer = memory)

In [50]:
graph.invoke({"messages":"hello my name is namir, what's your name?"},config = config)

{'messages': [HumanMessage(content="hello my name is namir, what's your name?", additional_kwargs={}, response_metadata={}, id='04b287a1-bf5f-4589-913c-5bff24a85268'),
  AIMessage(content=[{'type': 'text', 'text': "Hello Namir! It's nice to meet you. I am Gemini, a large language model built by Google. How can I help you today?", 'extras': {'signature': 'EuoFCucFAWkUfROsnU//6MC6cdScpRyKCp2TM+4SrYYTluitp1H+8cI2NDkqKBbdx0/eosz9xASqz+St44meSpORytfB1q7biBzBdsIoQbpFYkQ9B1qy5GLJw0kDuAQ8nTEsWgg9mYrknDDz6GtOcpk62SG78qjVD93/pDEErb3kqVaIcAW/2JudgvQC98z4FNx83MgXSFs6LFqMzdbgGB/JrjzbnALBojA/asiGe6jfwuncPo4vcL1UQ7dk75xQQHs/CRwTOfhWg1+xF7J9VqzrsEVDS5Uyv3x9uc6zzQyltLRaG0w1kOosxqjGd54jHUp0VKN6rt01mjgTDax4V7auTY3GsSuS+NGUEo/7usvrFWXcJDPOd7T6zseHxGBFF9SAiKxNYWDZM1ia209TXsgfBhx3skOnGZQNLZ75xy89psVUxQVDfOOTRKaVGCuVZKWVH7WE9J/PVh6VcGcJs1u4tmUHVKTWa8Yj+KuHkXYWPy309xRk2oG9q8Ukz9jn5CqAdSljT/C3d2HB+2crLwF6ANoEBmQT1t3PHLysN+telPPXl5gi5k0noJrqbHSEwdWDyFBZNHCxwnGnq3uh9QyRBfyd8uAJbusRFTVivwhNCrg5XKdZbERL9SFmhGJGvUa

In [53]:
graph.invoke({"messages":"okay gemini so you know my name right? do you know my surname"},config=config)

{'messages': [HumanMessage(content="hello my name is namir, what's your name?", additional_kwargs={}, response_metadata={}, id='04b287a1-bf5f-4589-913c-5bff24a85268'),
  AIMessage(content=[{'type': 'text', 'text': "Hello Namir! It's nice to meet you. I am Gemini, a large language model built by Google. How can I help you today?", 'extras': {'signature': 'EuoFCucFAWkUfROsnU//6MC6cdScpRyKCp2TM+4SrYYTluitp1H+8cI2NDkqKBbdx0/eosz9xASqz+St44meSpORytfB1q7biBzBdsIoQbpFYkQ9B1qy5GLJw0kDuAQ8nTEsWgg9mYrknDDz6GtOcpk62SG78qjVD93/pDEErb3kqVaIcAW/2JudgvQC98z4FNx83MgXSFs6LFqMzdbgGB/JrjzbnALBojA/asiGe6jfwuncPo4vcL1UQ7dk75xQQHs/CRwTOfhWg1+xF7J9VqzrsEVDS5Uyv3x9uc6zzQyltLRaG0w1kOosxqjGd54jHUp0VKN6rt01mjgTDax4V7auTY3GsSuS+NGUEo/7usvrFWXcJDPOd7T6zseHxGBFF9SAiKxNYWDZM1ia209TXsgfBhx3skOnGZQNLZ75xy89psVUxQVDfOOTRKaVGCuVZKWVH7WE9J/PVh6VcGcJs1u4tmUHVKTWa8Yj+KuHkXYWPy309xRk2oG9q8Ukz9jn5CqAdSljT/C3d2HB+2crLwF6ANoEBmQT1t3PHLysN+telPPXl5gi5k0noJrqbHSEwdWDyFBZNHCxwnGnq3uh9QyRBfyd8uAJbusRFTVivwhNCrg5XKdZbERL9SFmhGJGvUa

In [54]:
graph.invoke({"messages":"okay my surname is shah"},config=config)

{'messages': [HumanMessage(content="hello my name is namir, what's your name?", additional_kwargs={}, response_metadata={}, id='04b287a1-bf5f-4589-913c-5bff24a85268'),
  AIMessage(content=[{'type': 'text', 'text': "Hello Namir! It's nice to meet you. I am Gemini, a large language model built by Google. How can I help you today?", 'extras': {'signature': 'EuoFCucFAWkUfROsnU//6MC6cdScpRyKCp2TM+4SrYYTluitp1H+8cI2NDkqKBbdx0/eosz9xASqz+St44meSpORytfB1q7biBzBdsIoQbpFYkQ9B1qy5GLJw0kDuAQ8nTEsWgg9mYrknDDz6GtOcpk62SG78qjVD93/pDEErb3kqVaIcAW/2JudgvQC98z4FNx83MgXSFs6LFqMzdbgGB/JrjzbnALBojA/asiGe6jfwuncPo4vcL1UQ7dk75xQQHs/CRwTOfhWg1+xF7J9VqzrsEVDS5Uyv3x9uc6zzQyltLRaG0w1kOosxqjGd54jHUp0VKN6rt01mjgTDax4V7auTY3GsSuS+NGUEo/7usvrFWXcJDPOd7T6zseHxGBFF9SAiKxNYWDZM1ia209TXsgfBhx3skOnGZQNLZ75xy89psVUxQVDfOOTRKaVGCuVZKWVH7WE9J/PVh6VcGcJs1u4tmUHVKTWa8Yj+KuHkXYWPy309xRk2oG9q8Ukz9jn5CqAdSljT/C3d2HB+2crLwF6ANoEBmQT1t3PHLysN+telPPXl5gi5k0noJrqbHSEwdWDyFBZNHCxwnGnq3uh9QyRBfyd8uAJbusRFTVivwhNCrg5XKdZbERL9SFmhGJGvUa

In [55]:
graph.invoke({"messages":"okay i wanted to know about mayor mamdani"},config=config)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}